# Notebook 5 — The Acme Knowledge Assistant (RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikapv-kv/ai-upskill-layer-3/blob/main/notebooks/05_complete_rag_assistant.ipynb)

**Story so far:** we can retrieve the right runbook chunks (NB4). Now we give them to an **LLM** so it answers the developer's question, with sources.

**You'll build the full pipeline:** ingestion → chunking → embedding → indexing (from NB4) → **retrieval → augmentation → generation** → answer + sources.

⏱️ **In session (10 min):** ask the assistant, E5.1 · 🏠 **Take-home:** E5.2, the chat UI

⚡ **Runtime:** CPU is fine with the default Groq backend (needs a free API key, see §2). Without a key the local model is used: then pick *Runtime → Change runtime type → T4 GPU* (free).

## 0. Setup

In [ ]:
%pip install -q sentence-transformers==6.1.0 chromadb==1.5.9

In [ ]:
import json, os, re, urllib.request
import requests
import chromadb
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "Qwen/Qwen3-1.7B"                       # Apache-2.0, ~4 GB, runs locally, no API key
FALLBACK_LLM_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # smaller and faster, but much less reliable
MAX_NEW_TOKENS = 300

TOP_K = 3
CHUNK_SIZE = 120
CHUNK_OVERLAP = 20
COLLECTION_NAME = "acme_runbooks"
NO_ANSWER_TEXT = "The provided documentation does not contain enough information to answer this question."

def says_no_answer(text):
    """Small models sometimes reword the sentence slightly, so match its key phrase."""
    return "not contain enough information" in text

DATA_BASE_URL = "https://raw.githubusercontent.com/karthikapv-kv/ai-upskill-layer-3/main/data"

def data_file(relative_path):
    """Local path to a workshop data file; downloads it from GitHub if needed."""
    for folder in ["data", "../data"]:
        path = os.path.join(folder, relative_path)
        if os.path.exists(path):
            return path
    path = os.path.join("data", relative_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(f"{DATA_BASE_URL}/{relative_path}", path)
    return path

## 1. Ingestion, chunking, embedding, indexing *(carried over from Notebook 4)*

In [ ]:
embedder = SentenceTransformer(MODEL_NAME)

def embed_texts(texts):
    return embedder.encode(texts, normalize_embeddings=True)

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    for start in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
    return chunks

with open(data_file("runbooks/manifest.json")) as f:
    runbooks = json.load(f)
chunks = []
for runbook in runbooks:
    with open(data_file("runbooks/" + runbook["file"])) as f:
        for i, text in enumerate(chunk_text(f.read())):
            chunks.append({"chunk_id": f"{runbook['document_id']}::chunk-{i:02d}", "source_document": runbook["document_id"],
                           "title": runbook["title"], "category": runbook["category"], "text": text})

client = chromadb.EphemeralClient()
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME, configuration={"hnsw": {"space": "cosine"}}, embedding_function=None)
collection.add(ids=[c["chunk_id"] for c in chunks], documents=[c["text"] for c in chunks],
               metadatas=[{k: c[k] for k in ("source_document", "title", "category")} for c in chunks],
               embeddings=embed_texts([c["text"] for c in chunks]).tolist())

def retrieve(question, top_k=TOP_K):
    result = collection.query(query_embeddings=embed_texts([question]).tolist(), n_results=top_k)
    return [{"chunk_id": i, "text": t, **m, "similarity": round(1 - d, 3)}
            for i, t, m, d in zip(result["ids"][0], result["documents"][0], result["metadatas"][0], result["distances"][0])]

print("Index ready:", collection.count(), "chunks")

## 2. Load the LLM
Two backends, same pipeline. Pick one with `LLM_BACKEND`:
- **`"groq"`** (default): `openai/gpt-oss-20b` (Apache-2.0) on Groq's free tier. Fast, no GPU or download; needs a free API key from [console.groq.com](https://console.groq.com/keys).
- **`"local"`**: Qwen3-1.7B runs inside Colab. No key; ~4 GB download; T4 GPU recommended.

**Adding the key (once):** left sidebar **🔑 Secrets** → *Add new secret* → name `GROQ_API_KEY`, value = your key → switch on **Notebook access**. No secret? You'll be asked to paste the key into a hidden box; leave it empty to use the local model. Never type the key into a code cell.

If something fails we fall back to the local model, then the small model, then **prompt-only mode** (shows the prompt, no answer).

In [ ]:
LLM_BACKEND = "groq"      # "groq" (gpt-oss-20b, free API key) or "local" (Qwen3-1.7B in Colab)
USE_SMALL_MODEL = False   # local only: True = faster but much less reliable 0.5B model
CHAT_OPTIONS = {"enable_thinking": False}   # Qwen3 can "think" out loud first; we turn that off

GROQ_MODEL = "openai/gpt-oss-20b"           # Apache-2.0 open-weight model, hosted by Groq
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def get_groq_key():
    """Colab 🔑 Secrets first, then an environment variable, then a hidden prompt."""
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass
    if os.environ.get("GROQ_API_KEY"):
        return os.environ["GROQ_API_KEY"]
    from getpass import getpass
    return getpass("Paste your Groq API key (hidden, leave empty to use the local model): ").strip() or None

def load_llm():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    names = [FALLBACK_LLM_MODEL_NAME] if USE_SMALL_MODEL else [LLM_MODEL_NAME, FALLBACK_LLM_MODEL_NAME]
    for name in names:
        try:
            tokenizer = AutoTokenizer.from_pretrained(name)
            model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(device)
            print(f"Loaded {name} on {device}")
            return tokenizer, model
        except Exception as error:
            print(f"Could not load {name}: {error}")
    print("⚠️ No LLM available: running in prompt-only mode.")
    return None, None

tokenizer, llm = None, None
if LLM_BACKEND == "groq":
    GROQ_API_KEY = get_groq_key()
    if GROQ_API_KEY:
        print(f"Using {GROQ_MODEL} on Groq (no download needed)")
    else:
        print("⚠️ No GROQ_API_KEY found (add it in Colab's 🔑 Secrets panel). Using the local model instead.")
        LLM_BACKEND = "local"
if LLM_BACKEND == "local":
    tokenizer, llm = load_llm()

## 3. Augmentation: build the prompt
Instructions (system message) + retrieved chunks with their IDs + the question (user message).

In [ ]:
INSTRUCTIONS = f"""You are the Acme Pay internal engineering assistant.
Answer the question using ONLY the documentation excerpts below.
If the excerpts do not contain enough information, reply exactly:
"{NO_ANSWER_TEXT}"
Cite the chunk IDs you used in square brackets, e.g. [RB-01::chunk-03]."""

def build_context(chunks):
    return "\n\n".join(f"[{c['chunk_id']}] ({c['title']})\n{c['text']}" for c in chunks)

def build_prompt(question, context, instructions=INSTRUCTIONS):
    """The chat messages we send to the LLM."""
    return [
        {"role": "system", "content": instructions},
        {"role": "user", "content": f"Documentation excerpts:\n{context}\n\nQuestion: {question}\nAnswer:"},
    ]

Exactly what the LLM will receive for one question (no hidden magic):

In [ ]:
question = "How do I investigate a duplicate payment?"
messages = build_prompt(question, build_context(retrieve(question)))
if tokenizer is not None:
    print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_OPTIONS))
else:
    print(json.dumps(messages, indent=2))   # with Groq, exactly these messages are sent as JSON

**Expected:** a system block with our instructions, a user block with three `[RB-01::chunk-…]` excerpts and the question, then the assistant turn for the model to fill (the empty `<think></think>` is Qwen3's thinking mode, switched off).

## 4. Retrieval + generation: ask the assistant

In [ ]:
def call_groq(messages):
    """One HTTP POST to Groq's OpenAI-compatible endpoint."""
    response = requests.post(
        GROQ_URL,
        headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
        json={"model": GROQ_MODEL, "messages": messages, "temperature": 0,
              "max_completion_tokens": 1024, "reasoning_effort": "low", "include_reasoning": False},
        timeout=60,
    )
    if response.status_code == 429:
        return "(Groq rate limit reached: wait a minute and try again)"
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"].strip()

def call_llm(messages):
    if LLM_BACKEND == "groq":
        return call_groq(messages)
    if llm is None:
        return "(prompt-only mode: no LLM loaded)"
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                           return_dict=True, **CHAT_OPTIONS).to(llm.device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)   # greedy = repeatable
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def answer_question(question, top_k=TOP_K, instructions=INSTRUCTIONS, min_similarity=None):
    retrieved = retrieve(question, top_k)
    if min_similarity is not None and retrieved[0]["similarity"] < min_similarity:
        answer = NO_ANSWER_TEXT                        # guard: skip the LLM (section 7)
    else:
        answer = call_llm(build_prompt(question, build_context(retrieved), instructions))
    cited = re.findall(r"RB-\d\d::chunk-\d\d", answer)
    return {"answer": answer, "sources": [c["chunk_id"] for c in retrieved], "retrieved_chunks": retrieved, "cited": cited}

def show_answer(question, **options):
    result = answer_question(question, **options)
    print(f"🔎 {question}\n\n{result['answer']}\n\nSources (retrieved):")
    for c in result["retrieved_chunks"]:
        mark = "✔ cited" if c["chunk_id"] in result["cited"] else ""
        print(f"   {c['chunk_id']:<16} {c['similarity']:.3f}  {c['title']}  {mark}")
    print()
    return result

_ = show_answer("How do I roll back a failed deployment?")

**Expected:** steps with `acmectl history` and `acmectl rollback <service> --to <previous-version>`, grounded in RB-05. A few seconds on a T4 GPU; on CPU it can take a minute or more. (Long answers may be cut off at `MAX_NEW_TOKENS`.)

### ⏱️ Try the other developer questions

In [ ]:
for q in ["How do I investigate a duplicate payment?",
          "Our Kafka consumer stopped processing messages."]:
    _ = show_answer(q)

## 5. When RAG goes wrong: three cases
Answer exists · partial information · no information.

In [ ]:
with open(data_file("rag_test_questions.json")) as f:
    test_questions = json.load(f)

for case in test_questions[:3]:
    print(f"━━━ {case['case'].upper()} ━━━")
    _ = show_answer(case["question"])

**Expected** (Qwen3-1.7B; small models vary):
- **Answerable:** good steps from RB-01, citing `[RB-01::chunk-00]` and `[RB-01::chunk-03]` ✅
- **Partial:** "not enough information", although the refund timeline (5–7 business days) **is** in `RB-01::chunk-07`. Too cautious: safe, but unhelpful.
- **No information:** "not enough information" ✅, although retrieval still returned 3 chunks (best only ≈ **0.31**).

## 6. Without RAG

In [ ]:
print(call_llm([{"role": "user", "content": "How do I roll back a failed deployment at Acme Pay?"}]))

**Expected:** a confident, generic guide (Git, CI/CD tools, cloud consoles…). No `acmectl`, no sources: it **invents** a plausible process, because it has never seen Acme's runbooks.

### ⏱️ E5.1: A question the docs don't cover (~4 min)
Does the assistant admit it doesn't know? Check the similarity scores too.

In [ ]:
# ⏱️ E5.1: a question Acme's runbooks do NOT cover
e8_question = ""   # TODO

<details><summary>💡 Hint 1</summary>

Anything outside payments, Kafka, APIs, databases, deployments and incidents: *office Wi-Fi*, *holiday policy*, *Kubernetes*…

</details>

In [ ]:
if not e8_question:
    print("E5.1 not finished yet: write a question above.")
else:
    _ = show_answer(e8_question)

## 7. A safety net: skip the LLM when retrieval is weak 🏠
If even the best chunk scores low, don't ask the LLM at all. ⚠️ There's no universal threshold; this value is only for MiniLM on Acme's docs.

In [ ]:
MIN_SIMILARITY = 0.35

for q in ["How do I configure Kubernetes autoscaling?", "Can I deploy on Friday afternoon?"]:
    best = retrieve(q, top_k=1)[0]["similarity"]
    blocked = best < MIN_SIMILARITY
    print(f"{best:.3f}  {'BLOCKED' if blocked else 'sent to LLM'}  {q}")

**Expected:** Kubernetes **0.314** → blocked ✅. Friday **0.283** → blocked ❌, even though RB-05 answers it (no deploys after 15:00 on Fridays): *"afternoon"* ≠ *"15:00"*.

Useful as a safety net (the 0.5B model *invents* Kubernetes steps), but it can block good questions. Tune and monitor it; never trust it blindly.

## 8. Mini evaluation 🏠

In [ ]:
rows = []
for case in test_questions:
    result = answer_question(case["question"])
    retrieved_docs = {c["source_document"] for c in result["retrieved_chunks"]}
    rows.append({
        "case": case["case"],
        "question": case["question"][:55],
        "expected source retrieved": (set(case["expected_sources"]) <= retrieved_docs) if case["expected_sources"] else "n/a",
        "cited a chunk": bool(result["cited"]),
        "said 'not enough info'": says_no_answer(result["answer"]),
    })

import pandas as pd
pd.DataFrame(rows)

**Look for:** retrieval found the right runbook **every** time, yet the LLM cited sources only twice and said "not enough info" for 3 questions the docs do answer (API slow, Friday, partial). Retrieval and generation fail **independently**.

## 🏠 E5.2: Better source attribution
Improve the instructions so every claim is traceable, then compare.

In [ ]:
# 🏠 E5.2: edit the instructions so answers cite their sources more reliably
IMPROVED_INSTRUCTIONS = INSTRUCTIONS   # TODO: change this text

<details><summary>💡 Hint 1</summary>

Ask for the chunk ID after *every* sentence and a final `Sources:` line.

</details>

<details><summary>💡 Hint 2</summary>

Also ask it to say which part of a question isn't covered (helps the partial case).

</details>

In [ ]:
for q in ["How do I investigate a duplicate payment?",
          "How long does a duplicate-charge refund take, and who must approve refunds above $10,000?"]:
    _ = show_answer(q, instructions=IMPROVED_INSTRUCTIONS)

## 🏠 Optional: a chat UI
A tiny web UI around `answer_question`. Set `LAUNCH_UI = True` to open it in the notebook.

In [ ]:
%pip install -q gradio==6.28.0

In [ ]:
import gradio as gr

def assistant(question):
    result = answer_question(question)
    sources = "\n".join(f"- `{c['chunk_id']}` {c['title']} ({c['similarity']:.3f})" for c in result["retrieved_chunks"])
    return f"{result['answer']}\n\n**Sources**\n{sources}"

ui = gr.Interface(fn=assistant, inputs=gr.Textbox(label="Ask the Acme Knowledge Assistant"),
                  outputs=gr.Markdown(), title="Acme Knowledge Assistant")

LAUNCH_UI = False
if LAUNCH_UI:
    ui.launch()

## Summary
- **RAG pipeline** = document ingestion → chunking → embedding → indexing → **retrieval → augmentation → generation**, returning the answer with sources.
- It **grounds** answers but doesn't guarantee them: watch for invented details, missing citations and wrong "I don't know"s.
- Retrieval quality limits answer quality.
- In production only `call_llm()` changes (bigger model, same pipeline). Frameworks like LangChain wrap these same steps.

🎉 You've built the Acme Knowledge Assistant: **question → embedding → vector search → context → prompt → LLM → answer + sources.**

---
## Solutions
**E5.1:** e.g. *"What is the office Wi-Fi password?"*: expect the "not enough information" sentence and low similarity scores. Retrieval still returns 3 unrelated chunks.

**E5.2:** with the improved instructions, every sentence gets a `[RB-01::chunk-03]` citation plus a `Sources:` line ✅. The partial case now answers the timeline (5–7 days cards, 1–2 wallets) ✅, but then **invents** an approver (*"the bank's compliance team"*) ❌. A prompt that makes the model more helpful can also make it more wrong, so always check the output against the sources.

In [ ]:
#@title E5.2 solution
IMPROVED_INSTRUCTIONS = INSTRUCTIONS + """
After EVERY sentence, add the chunk ID it came from in square brackets.
End with a line "Sources:" listing every chunk ID you used.
If part of the question is not covered by the excerpts, say which part."""
_ = show_answer("How do I investigate a duplicate payment?", instructions=IMPROVED_INSTRUCTIONS)